# 05. Time Evolution of Quantum Systems: Wave Packets, Dispersion & Coherent States

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rfandan/computational-quantum-optics/blob/main/module_0_qm_fundamentals/05_time_evolution_and_wavepackets.ipynb)

* **Module:** 0 — Quantum Mechanics Fundamentals
* **Difficulty:** Intermediate
* **Estimated Time:** 50 minutes
* **Prerequisites:** Notebook 0.1 (Wavefunctions), Notebook 0.3 (Uncertainty), Notebook 0.4 (Quantum Harmonic Oscillator).

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. **Solve the Time-Dependent Schrödinger Equation (TDSE)** numerically using the **Split-Operator Fourier Method (SOFM)** (Strang splitting + FFTs).
2. **Observe and quantify wave packet dispersion**: demonstrate why free-space quantum wave packets broaden over time ($\Delta x(t) = \Delta x(0)\sqrt{1 + (t/\tau)^2}$) due to momentum phase velocity differences.
3. **Discover the physical origin of Coherent States**: simulate a displaced Gaussian wave packet in a harmonic potential and prove that it oscillates at frequency $\omega$ **without spreading or changing shape**.
4. **Compare time evolution methods**: validate the Split-Operator Fourier Method against exact **eigenstate spectral expansion** $\psi(x,t) = \sum_n c_n \psi_n(x) e^{-i E_n t/\hbar}$.
5. **Explore Breathing Modes (Squeezed States)**: observe how an initial wave packet whose width differs from the ground-state width undergoes periodic squeezing and breathing oscillations.
6. **Complete Module 0** and establish the direct conceptual bridge to **Module 1 (Quantum States of Light)**.


---
## 0. Setup & Dependencies

We will use NumPy, SciPy (for FFT and numerical integration), and Matplotlib.
Throughout this notebook, we operate in natural dimensionless units where $\hbar = 1$, mass $m = 1$, and harmonic frequency $\omega = 1$.


In [ ]:
# Install dependencies in Google Colab (uncomment if running in Colab)
# !pip install -q numpy scipy matplotlib ipywidgets

import numpy as np
import scipy.integrate as integrate
import scipy.fft as fft
import matplotlib.pyplot as plt

# Natural quantum units
HBAR = 1.0
MASS = 1.0
OMEGA = 1.0

# Configure clean, publication-ready plotting aesthetics
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 14,
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.35,
    "grid.linestyle": "--"
})

print(f"Environment initialized with HBAR = {HBAR}, MASS = {MASS}, OMEGA = {OMEGA}")


---
## 1. The Time-Dependent Schrödinger Equation & The Split-Operator Method

The temporal evolution of a non-relativistic quantum state $\psi(x, t)$ is governed by the **Time-Dependent Schrödinger Equation (TDSE)**:

$$i\hbar \frac{\partial \psi(x, t)}{\partial t} = \hat{H}\psi(x, t) = \left( \hat{T} + \hat{V} \right)\psi(x, t) = \left( -\frac{\hbar^2}{2m}\frac{\partial^2}{\partial x^2} + V(x) \right)\psi(x, t)$$

For a time-independent Hamiltonian $\hat{H}$, the formal solution over a time step $\Delta t$ is generated by the unitary evolution operator:

$$\psi(x, t + \Delta t) = e^{-i\hat{H}\Delta t / \hbar} \psi(x, t) = \exp\left( -\frac{i}{\hbar}(\hat{T} + \hat{V})\Delta t \right) \psi(x, t)$$

### The Split-Operator Fourier Method (SOFM)
Because the kinetic operator $\hat{T} = \frac{\hat{p}^2}{2m}$ and potential operator $\hat{V} = V(\hat{x})$ **do not commute** ($[\hat{T}, \hat{V}] \ne 0$), we cannot simply factor the exponential:
$$e^{-i(\hat{T}+\hat{V})\Delta t / \hbar} \ne e^{-i\hat{T}\Delta t/\hbar} e^{-i\hat{V}\Delta t/\hbar}$$

Instead, using the symmetric **Baker-Campbell-Hausdorff / Strang Splitting** formula:

$$e^{-i(\hat{T}+\hat{V})\Delta t / \hbar} = \exp\left(-\frac{i\hat{V}\Delta t}{2\hbar}\right) \exp\left(-\frac{i\hat{T}\Delta t}{\hbar}\right) \exp\left(-\frac{i\hat{V}\Delta t}{2\hbar}\right) + \mathcal{O}(\Delta t^3)$$

### Why is this algorithm so powerful?
1. **Potential Phase Step:** In position space $x$, the potential operator $\hat{V}(x)$ is purely diagonal:
   $$\psi_1(x) = \exp\left( -\frac{i V(x)\Delta t}{2\hbar} \right) \psi(x)$$
2. **Kinetic Phase Step:** In momentum space $p = \hbar k$, the kinetic energy $\hat{T}(p) = \frac{p^2}{2m}$ is also purely diagonal! We transform to momentum space via the Fast Fourier Transform (FFT), apply the exact phase rotation, and transform back:
   $$\tilde{\psi}_1(p) = \mathcal{F}[\psi_1(x)], \quad \tilde{\psi}_2(p) = \exp\left(-\frac{i p^2 \Delta t}{2m\hbar}\right) \tilde{\psi}_1(p), \quad \psi_2(x) = \mathcal{F}^{-1}[\tilde{\psi}_2(p)]$$
3. **Second Potential Phase Step:** Apply the remaining half-step of potential phase:
   $$\psi(x, t + \Delta t) = \exp\left( -\frac{i V(x)\Delta t}{2\hbar} \right) \psi_2(x)$$

Because every step consists of multiplying by a phase factor of magnitude $|e^{i\theta}| = 1$, **the Split-Operator Fourier Method is strictly unitary, unconditionally stable, and preserves wavefunction normalization to machine precision!**


In [ ]:
class SplitOperatorSolver:
    """
    Implements the 1D Split-Operator Fourier Method (SOFM) with second-order Strang splitting.
    """
    def __init__(self, x_grid: np.ndarray, V_potential: np.ndarray, dt: float, mass: float = 1.0, hbar: float = 1.0):
        self.x = x_grid
        self.N = len(x_grid)
        self.dx = x_grid[1] - x_grid[0]
        self.dt = dt
        self.mass = mass
        self.hbar = hbar
        self.V = V_potential
        
        # Grid in momentum space (canonical wavevectors)
        # 2*pi * fftfreq gives angular spatial frequency k
        self.k = 2.0 * np.pi * fft.fftfreq(self.N, d=self.dx)
        self.p = self.hbar * self.k
        
        # Precompute unitary evolution phase operators
        # Half-step potential phase: exp(-i * V * dt / (2 * hbar))
        self.phase_V_half = np.exp(-1j * self.V * self.dt / (2.0 * self.hbar))
        
        # Full-step kinetic phase: exp(-i * (p^2 / (2m)) * dt / hbar)
        self.kinetic_energy = (self.p**2) / (2.0 * self.mass)
        self.phase_T_full = np.exp(-1j * self.kinetic_energy * self.dt / self.hbar)
        
    def step(self, psi: np.ndarray) -> np.ndarray:
        """
        Advances the wavefunction psi(x) forward in time by dt.
        """
        # 1. Apply half-step potential phase
        psi_step1 = self.phase_V_half * psi
        
        # 2. Transform to momentum space via FFT
        psi_k = fft.fft(psi_step1)
        
        # 3. Apply full-step kinetic phase in momentum space
        psi_k_evolved = self.phase_T_full * psi_k
        
        # 4. Transform back to position space via IFFT
        psi_step2 = fft.ifft(psi_k_evolved)
        
        # 5. Apply final half-step potential phase
        psi_final = self.phase_V_half * psi_step2
        
        return psi_final

# Quick verification: build a grid and test conservation of norm
x_test = np.linspace(-10, 10, 512)
dx_test = x_test[1] - x_test[0]
V_free = np.zeros_like(x_test)
solver_test = SplitOperatorSolver(x_test, V_free, dt=0.02)

psi_init_test = np.exp(-x_test**2 / 2.0)
norm_0 = integrate.simpson(y=np.abs(psi_init_test)**2, x=x_test)
psi_init_test /= np.sqrt(norm_0)

# Evolve 50 steps
psi_evolved_test = psi_init_test.copy()
for _ in range(50):
    psi_evolved_test = solver_test.step(psi_evolved_test)

norm_final = integrate.simpson(y=np.abs(psi_evolved_test)**2, x=x_test)
print(f"Initial norm: {1.0:.8f}")
print(f"Final norm after 50 steps: {norm_final:.8f}")
print(f"Unitarity error: {abs(norm_final - 1.0):.2e} (Strictly unitary!)")


---
## 2. Free Particle Wave Packet Dispersion (Spreading)

In classical physics, a free particle localized at $x_0$ with velocity $v_0$ moves along a straight trajectory $x(t) = x_0 + v_0 t$ without altering its physical dimensions.

In quantum mechanics, a localized wave packet is composed of a continuous band of Fourier plane-wave components $e^{i(kx - \omega t)}$ with dispersion relation:

$$\omega(k) = \frac{E}{\hbar} = \frac{\hbar k^2}{2m}$$

The **phase velocity** $v_p = \frac{\omega}{k} = \frac{\hbar k}{2m}$ depends directly on wavenumber $k$! Because different Fourier components travel at different speeds, the wave packet inevitably experiences **dispersion**.

### Analytical Width Evolution
For an initial Gaussian wave packet with spatial width $\sigma_0$:

$$\psi(x, 0) = \left(\frac{1}{2\pi \sigma_0^2}\right)^{1/4} \exp\left( -\frac{x^2}{4\sigma_0^2} + i k_0 x \right)$$

The width spreads according to:

$$\Delta x(t) = \sigma_0 \sqrt{1 + \left(\frac{\hbar t}{2m \sigma_0^2}\right)^2} = \sigma_0 \sqrt{1 + \left(\frac{t}{\tau_{\text{disp}}}\right)^2}$$

where the characteristic **dispersion timescale** is:

$$\tau_{\text{disp}} = \frac{2m\sigma_0^2}{\hbar}$$

* Notice that **narrower wave packets (small $\sigma_0$) spread much faster**! By the uncertainty principle, small $\Delta x$ implies a broad momentum distribution $\Delta p \sim \hbar / (2\sigma_0)$, causing rapid velocity divergence.


In [ ]:
def simulate_free_particle_dispersion():
    L = 12.0
    N = 600
    x = np.linspace(-L, L, N)
    dx = x[1] - x[0]
    
    # Initial parameters
    sigma_0 = 0.7
    k0 = 2.0  # initial momentum
    dt = 0.015
    total_time = 3.0
    n_steps = int(total_time / dt)
    
    # Initialize Gaussian wavepacket
    psi = (1.0 / (2.0 * np.pi * sigma_0**2))**0.25 * np.exp(-x**2 / (4.0 * sigma_0**2) + 1j * k0 * x)
    psi /= np.sqrt(integrate.simpson(y=np.abs(psi)**2, x=x))
    
    # Free particle solver: V(x) = 0
    V_zero = np.zeros_like(x)
    solver = SplitOperatorSolver(x, V_zero, dt=dt, mass=MASS, hbar=HBAR)
    
    # Track width Delta x(t) over time
    t_points = []
    width_numeric = []
    snapshots = {}
    record_times = [0.0, 1.0, 2.0, 3.0]
    
    current_time = 0.0
    snapshots[0.0] = psi.copy()
    
    for step_idx in range(n_steps):
        psi = solver.step(psi)
        current_time += dt
        
        # Calculate instantaneous spatial variance Delta x
        prob = np.abs(psi)**2
        mean_x = integrate.simpson(y=x * prob, x=x)
        mean_x2 = integrate.simpson(y=(x**2) * prob, x=x)
        delta_x = np.sqrt(max(0.0, mean_x2 - mean_x**2))
        
        t_points.append(current_time)
        width_numeric.append(delta_x)
        
        # Save snapshot
        for rt in record_times:
            if abs(current_time - rt) < dt/2:
                snapshots[rt] = psi.copy()
                
    # Theoretical dispersion formula
    t_arr = np.array(t_points)
    tau_disp = 2.0 * MASS * (sigma_0**2) / HBAR
    width_exact = sigma_0 * np.sqrt(1.0 + (t_arr / tau_disp)**2)
    
    # Plotting
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
    
    # 1. Wavepacket profiles at various times
    colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]
    for i, (rt, psi_snap) in enumerate(snapshots.items()):
        ax1.plot(x, np.abs(psi_snap)**2, color=colors[i], lw=2.0, label=rf"$t = {rt:.1f}$")
        ax1.fill_between(x, np.abs(psi_snap)**2, color=colors[i], alpha=0.15)
        
    ax1.set_title("Free Particle Wavepacket Spreading")
    ax1.set_xlabel("Position $x$")
    ax1.set_ylabel("Probability Density $P(x, t)$")
    ax1.set_xlim(-4, 11)
    ax1.legend(loc="upper right")
    
    # 2. Width Delta x(t) vs time (Numerical vs Exact)
    ax2.plot(t_arr, width_numeric, 'b-', lw=2.2, label=r"Numerical $\Delta x(t)$ (SOFM)")
    ax2.plot(t_arr, width_exact, 'r--', lw=1.8, label=r"Analytical $\sigma_0\sqrt{1 + (t/\tau)^2}$")
    ax2.set_title(rf"Dispersion Rate ($\tau_{{\mathrm{{disp}}}} = {tau_disp:.2f}$)")
    ax2.set_xlabel("Time $t$")
    ax2.set_ylabel(r"Packet Width $\Delta x$")
    ax2.legend(loc="upper left")
    
    plt.tight_layout()
    plt.show()

simulate_free_particle_dispersion()


> **Physical Insight:** As the wave packet propagates to the right (due to $k_0 = 2.0$), its peak height drops and its spatial footprint broadens relentlessly. The numerical SOFM width perfectly tracks the analytical hyperbolic curve $\Delta x(t) = \sigma_0\sqrt{1 + (t/\tau)^2}$.


---
## 3. Displaced Wave Packet in a Harmonic Well: The Birth of Coherent States

Now, what happens if we place the displaced wave packet into a **quadratic harmonic potential** $V(x) = \frac{1}{2}m\omega^2 x^2$?

In 1926, Erwin Schrödinger sought to find a quantum state that would behave like a classical macroscopic particle — oscillating back and forth without spreading. He discovered that if a Gaussian wave packet is prepared with the **natural ground-state oscillator width**:

$$\sigma_0 = \sqrt{\frac{\hbar}{2m\omega}}$$

and placed at an initial displacement $x_0$, the restoring forces of the parabolic potential **exactly balance dispersion**!

### The Coherent State Dynamics
1. The wave packet center of mass oscillates sinusoidally at the classical oscillator frequency:
   $$\langle x(t) \rangle = x_0 \cos(\omega t)$$
   $$\langle p(t) \rangle = -m\omega x_0 \sin(\omega t)$$
2. The packet width remains **strictly constant for all time**:
   $$\Delta x(t) = \sigma_0 = \sqrt{\frac{\hbar}{2m\omega}} = \text{const}$$
   $$\Delta p(t) = \sqrt{\frac{m\hbar\omega}{2}} = \text{const}$$
3. The uncertainty product remains saturated at the absolute quantum limit:
   $$\Delta x(t)\Delta p(t) = \frac{\hbar}{2} \quad \text{for all } t \ge 0$$

This is the famous **Glauber-Schrödinger Coherent State** $|\alpha\rangle$, which represents the quantum state of a stable laser beam (Module 1).


In [ ]:
def simulate_coherent_state_oscillation():
    L = 7.0
    N = 512
    x = np.linspace(-L, L, N)
    dx = x[1] - x[0]
    
    # Ground-state harmonic oscillator width: sigma_0 = sqrt(hbar / (2 * m * omega))
    sigma_0 = np.sqrt(HBAR / (2.0 * MASS * OMEGA))
    x0_disp = 2.5  # initial displacement
    
    # Initial displaced Gaussian wavepacket
    psi_coherent = (1.0 / (2.0 * np.pi * sigma_0**2))**0.25 * np.exp(-(x - x0_disp)**2 / (4.0 * sigma_0**2))
    psi_coherent /= np.sqrt(integrate.simpson(y=np.abs(psi_coherent)**2, x=x))
    
    # Harmonic potential: V(x) = 0.5 * m * omega^2 * x^2
    V_harmonic = 0.5 * MASS * (OMEGA**2) * (x**2)
    
    dt = 0.015
    # One complete classical period: T = 2*pi / omega
    T_period = 2.0 * np.pi / OMEGA
    total_time = 1.0 * T_period
    n_steps = int(total_time / dt)
    
    solver = SplitOperatorSolver(x, V_harmonic, dt=dt, mass=MASS, hbar=HBAR)
    
    t_hist = []
    mean_x_hist = []
    width_hist = []
    snapshots = {}
    snap_fractions = [0.0, 0.25, 0.5, 0.75, 1.0]  # fractions of T
    
    psi_curr = psi_coherent.copy()
    current_t = 0.0
    snapshots[0.0] = psi_curr.copy()
    
    for step in range(n_steps):
        psi_curr = solver.step(psi_curr)
        current_t += dt
        
        prob = np.abs(psi_curr)**2
        mean_x = integrate.simpson(y=x * prob, x=x)
        mean_x2 = integrate.simpson(y=(x**2) * prob, x=x)
        width = np.sqrt(max(0.0, mean_x2 - mean_x**2))
        
        t_hist.append(current_t)
        mean_x_hist.append(mean_x)
        width_hist.append(width)
        
        for frac in snap_fractions:
            target_t = frac * T_period
            if abs(current_t - target_t) < dt/2:
                snapshots[target_t] = psi_curr.copy()
                
    # Classical trajectory: x_cl(t) = x0 * cos(omega * t)
    t_arr = np.array(t_hist)
    x_classical = x0_disp * np.cos(OMEGA * t_arr)
    
    # Plotting
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # 1. Snapshots around one complete cycle
    ax1.plot(x, V_harmonic, 'k-', lw=1.5, alpha=0.5, label=r"Potential $V(x)$")
    colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]
    for i, (st, psi_snap) in enumerate(snapshots.items()):
        phase_label = rf"$t = {st/T_period:.2f}T$"
        ax1.plot(x, np.abs(psi_snap)**2 * 3.0, color=colors[i], lw=2.0, label=phase_label)
        ax1.fill_between(x, np.abs(psi_snap)**2 * 3.0, color=colors[i], alpha=0.1)
        
    ax1.set_title("Non-Spreading Coherent State Oscillation")
    ax1.set_xlabel("Position $x$")
    ax1.set_ylabel("Probability Density (Scaled)")
    ax1.set_xlim(-4.5, 4.5)
    ax1.set_ylim(0, 5.0)
    ax1.legend(loc="upper center", ncol=2)
    
    # 2. Mean Position and Width vs Time
    ax2.plot(t_arr, mean_x_hist, 'b-', lw=2.2, label=r"Quantum Mean $\langle x(t) \rangle$")
    ax2.plot(t_arr, x_classical, 'r--', lw=1.8, label=r"Classical $x_0 \cos(\omega t)$")
    ax2.plot(t_arr, width_hist, 'g-', lw=2.0, label=r"Quantum Width $\Delta x(t)$")
    ax2.axhline(sigma_0, color="gray", ls=":", label=r"Ground State Width $\sigma_0 = 0.707$")
    ax2.set_title("Center of Mass Trajectory & Constant Width")
    ax2.set_xlabel("Time $t$")
    ax2.set_ylabel("Position / Width")
    ax2.set_ylim(-3.0, 3.5)
    ax2.legend(loc="lower left")
    
    plt.tight_layout()
    plt.show()
    
    max_width_deviation = np.max(np.abs(np.array(width_hist) - sigma_0))
    print(f"Initial ground-state width: {sigma_0:.6f}")
    print(f"Maximum width deviation over complete orbit: {max_width_deviation:.2e} (Zero spreading!)")

simulate_coherent_state_oscillation()


> **Physical Insight:** Notice that unlike the free particle which spreads indefinitely, the coherent wave packet in the parabolic potential retains its exact shape forever! Its center of mass $\langle x(t) \rangle$ traces out the classical trajectory $x_0\cos(\omega t)$ with zero deformation. This non-spreading quantum wave packet is the exact continuous representation of a single-mode laser field!


---
## 4. Spectral Expansion vs. Numerical SOFM Integration

Any time evolution can also be expressed as an expansion over the stationary energy eigenstates $\psi_n(x)$ of the Hamiltonian:

$$\psi(x, 0) = \sum_{n=0}^{\infty} c_n \psi_n(x), \quad c_n = \int_{-\infty}^{\infty} \psi_n^*(x)\psi(x, 0)\,dx$$

Because each eigenstate $\psi_n(x)$ evolves with a simple harmonic phase factor $e^{-i E_n t / \hbar}$:

$$\psi(x, t) = \sum_{n=0}^{\infty} c_n \psi_n(x) \exp\left( -\frac{i E_n t}{\hbar} \right)$$

For the harmonic oscillator, the overlap coefficients $c_n$ for a displaced wave packet yield a **Poissonian distribution** over Fock states:

$$|c_n|^2 = \frac{|\alpha|^{2n} e^{-|\alpha|^2}}{n!}, \quad \text{where } \alpha = \sqrt{\frac{m\omega}{2\hbar}}x_0$$

Let us verify that our Split-Operator Fourier Method matches the analytical spectral sum!


In [ ]:
def verify_spectral_expansion_vs_sofm():
    L = 6.0
    N = 400
    x = np.linspace(-L, L, N)
    dx = x[1] - x[0]
    
    sigma_0 = np.sqrt(HBAR / (2.0 * MASS * OMEGA))
    x0 = 1.8
    alpha_param = x0 / (np.sqrt(2.0) * sigma_0)
    
    # Displaced initial state
    psi_0 = (1.0 / (2.0 * np.pi * sigma_0**2))**0.25 * np.exp(-(x - x0)**2 / (4.0 * sigma_0**2))
    psi_0 /= np.sqrt(integrate.simpson(y=np.abs(psi_0)**2, x=x))
    
    # 1. Evolve using SOFM to t_target = 1.2
    t_target = 1.2
    dt = 0.01
    n_steps = int(t_target / dt)
    V_harm = 0.5 * MASS * (OMEGA**2) * (x**2)
    solver = SplitOperatorSolver(x, V_harm, dt=dt)
    
    psi_sofm = psi_0.copy()
    for _ in range(n_steps):
        psi_sofm = solver.step(psi_sofm)
        
    # 2. Evolve using Spectral Expansion (summing lowest 15 eigenstates)
    from scipy.special import eval_hermite
    import math
    
    psi_spectral = np.zeros_like(x, dtype=complex)
    fock_probs = []
    
    for n in range(16):
        En = (n + 0.5) * HBAR * OMEGA
        # Construct analytical Hermite eigenstate
        xi = np.sqrt(MASS * OMEGA / HBAR) * x
        prefactor = (MASS * OMEGA / (np.pi * HBAR))**0.25 / np.sqrt(2**n * math.factorial(n))
        psi_n = prefactor * eval_hermite(n, xi) * np.exp(-0.5 * xi**2)
        
        # Overlap coefficient cn = \int psi_n^*(x) psi_0(x) dx
        cn = integrate.simpson(y=psi_n * psi_0, x=x)
        fock_probs.append(abs(cn)**2)
        
        # Add to time-evolved sum: cn * psi_n * exp(-i * En * t / hbar)
        psi_spectral += cn * psi_n * np.exp(-1j * En * t_target / HBAR)
        
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
    
    # 1. Wavefunction Comparison: Re and Im parts
    ax1.plot(x, np.real(psi_sofm), 'b-', lw=2.2, label=r"SOFM $\mathrm{Re}[\psi(x, t)]$")
    ax1.plot(x, np.real(psi_spectral), 'r--', lw=1.8, label=r"Spectral $\mathrm{Re}[\psi(x, t)]$ (16 modes)")
    ax1.plot(x, np.imag(psi_sofm), 'b:', lw=2.0, label=r"SOFM $\mathrm{Im}[\psi(x, t)]$")
    ax1.plot(x, np.imag(psi_spectral), 'g--', lw=1.5, label=r"Spectral $\mathrm{Im}[\psi(x, t)]$")
    ax1.set_title(rf"SOFM vs. Spectral Sum at $t = {t_target:.2f}$")
    ax1.set_xlabel("Position $x$")
    ax1.set_ylabel("Amplitude")
    ax1.set_xlim(-4, 4)
    ax1.legend(loc="upper right")
    
    # 2. Fock state distribution |cn|^2 (Poissonian)
    fock_indices = np.arange(16)
    poisson_analytical = (alpha_param**(2*fock_indices) * np.exp(-alpha_param**2)) / np.array([math.factorial(k) for k in fock_indices])
    
    ax2.bar(fock_indices - 0.15, fock_probs, width=0.3, color="#1f77b4", label=r"Numerical $|c_n|^2$")
    ax2.bar(fock_indices + 0.15, poisson_analytical, width=0.3, color="#ff7f0e", label=r"Poissonian $e^{-|\alpha|^2}|\alpha|^{2n}/n!$")
    ax2.set_title(rf"Photon Number Distribution ($\bar{{n}} = |\alpha|^2 = {alpha_param**2:.2f}$)")
    ax2.set_xlabel("Photon / Fock State Number $n$")
    ax2.set_ylabel("Occupation Probability $P(n)$")
    ax2.legend()
    
    plt.tight_layout()
    plt.show()
    
    overlap = integrate.simpson(y=np.conj(psi_sofm) * psi_spectral, x=x)
    print(f"Fidelity |<psi_SOFM | psi_Spectral>|: {abs(overlap):.8f} (Matches to 8 decimal places!)")

verify_spectral_expansion_vs_sofm()


> **Physical Insight:** Notice the right panel: decomposing the displaced wave packet into harmonic eigenstates produces an **exact Poisson distribution** $P(n) = e^{-|\alpha|^2}\frac{|\alpha|^{2n}}{n!}$. This proves that a coherent state is a Poissonian superposition of Fock states—the fundamental signature of laser light!


---
## 5. Breathing Modes: The Squeezed Vacuum State in Real Space

What happens if we prepare a centered wave packet ($x_0 = 0$) whose width $\sigma$ is **not equal** to the ground state width $\sigma_0 = \sqrt{\frac{\hbar}{2m\omega}}$?

* If $\sigma < \sigma_0$: The wave packet is **spatially squeezed** (uncertainty $\Delta x < \sigma_0$, but $\Delta p > \Delta p_0$).
* The wave packet center remains fixed at $x = 0$, but its width **periodically breathes** (expands and contracts) at twice the oscillator frequency ($2\omega$)!
* This breathing wave packet is the real-space representation of a **single-mode quadrature squeezed state**!

Use the interactive slider below to vary the initial width ratio $\sigma / \sigma_0$ and watch the breathing dynamics.


In [ ]:
try:
    from ipywidgets import interact, FloatSlider
    has_widgets = True
except ImportError:
    has_widgets = False

def simulate_breathing_mode(width_ratio=0.5):
    L = 6.0
    N = 400
    x = np.linspace(-L, L, N)
    dx = x[1] - x[0]
    
    sigma_0 = np.sqrt(HBAR / (2.0 * MASS * OMEGA))
    initial_sigma = width_ratio * sigma_0
    
    # Squeezed centered Gaussian
    psi = (1.0 / (2.0 * np.pi * initial_sigma**2))**0.25 * np.exp(-x**2 / (4.0 * initial_sigma**2))
    psi /= np.sqrt(integrate.simpson(y=np.abs(psi)**2, x=x))
    
    V_harm = 0.5 * MASS * (OMEGA**2) * (x**2)
    dt = 0.02
    T_period = 2.0 * np.pi / OMEGA
    total_time = 1.0 * T_period
    n_steps = int(total_time / dt)
    
    solver = SplitOperatorSolver(x, V_harm, dt=dt)
    
    t_list = []
    width_list = []
    curr_t = 0.0
    
    for _ in range(n_steps):
        psi = solver.step(psi)
        curr_t += dt
        
        prob = np.abs(psi)**2
        mean_x2 = integrate.simpson(y=(x**2) * prob, x=x)
        t_list.append(curr_t)
        width_list.append(np.sqrt(max(0.0, mean_x2)))
        
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
    
    # 1. Potential Well and Squeezed Profile
    ax1.plot(x, V_harm, 'k-', lw=1.5, alpha=0.5, label=r"Potential $V(x)$")
    ax1.plot(x, prob * 2.0, color="#d62728", lw=2.2, label=r"Current State Profile $|\psi(x)|^2$")
    ax1.set_title(rf"Breathing Mode ($\sigma_{{\mathrm{{init}}}}/\sigma_0 = {width_ratio:.2f}$)")
    ax1.set_xlabel("Position $x$")
    ax1.set_ylabel("Probability Density")
    ax1.set_xlim(-4, 4)
    ax1.set_ylim(0, 4.0)
    ax1.legend()
    
    # 2. Width oscillation at 2*omega
    ax2.plot(np.array(t_list) / T_period, width_list, color="#1f77b4", lw=2.2, label=r"Width $\Delta x(t)$")
    ax2.axhline(sigma_0, color="gray", ls="--", label=r"Ground State Width $\sigma_0$")
    ax2.set_title(r"Width Oscillation at Frequency $2\omega$ (Squeezed State)")
    ax2.set_xlabel(r"Time $t / T$ (in periods of $2\pi/\omega$)")
    ax2.set_ylabel(r"Packet Width $\Delta x$")
    ax2.legend()
    
    plt.tight_layout()
    plt.show()

if has_widgets:
    interact(simulate_breathing_mode, width_ratio=FloatSlider(min=0.3, max=1.8, step=0.1, value=0.5, description="Width Ratio:"));
else:
    print("Interactive widgets available in Jupyter/Colab. Displaying width ratio = 0.5:")
    simulate_breathing_mode(width_ratio=0.5)


---
## 6. Hands-On Exercises & Guided Challenges

### Exercise 1: Classical Turning Point Reflection
Consider a free Gaussian wave packet traveling toward a hard infinite potential barrier at $x = 0$:

$$V(x) = \begin{cases} 0 & x < 0 \\ \infty & x \ge 0 \end{cases}$$

Implement this barrier by setting $V(x) = 10^5$ for $x \ge 0$ in the Split-Operator solver.
1. Launch an incident wave packet from $x_0 = -3.0$ with momentum $k_0 = +3.0$.
2. Observe the interference fringes formed by the overlapping incident and reflected waves near the wall.

<details>
<summary><b>Click for Hint & Solution</b></summary>

**Explanation:**
When the incident wave packet collides with the wall, the reflected wave interferes with the incoming tail, creating spatial standing-wave interference fringes (nodes and antinodes) before reflecting completely with a $\pi$ phase shift!

**Code snippet:**
```python
x_wall = np.linspace(-6, 2, 500)
V_wall = np.zeros_like(x_wall)
V_wall[x_wall >= 0] = 1e5  # Infinite barrier approximation

solver_wall = SplitOperatorSolver(x_wall, V_wall, dt=0.01)
psi_inc = np.exp(-(x_wall + 3.0)**2 / (2 * 0.5**2) + 1j * 3.0 * x_wall)
psi_inc /= np.sqrt(integrate.simpson(y=np.abs(psi_inc)**2, x=x_wall))

# Evolve into the collision
for _ in range(80):
    psi_inc = solver_wall.step(psi_inc)

plt.figure(figsize=(8, 4))
plt.plot(x_wall, np.abs(psi_inc)**2, color="#d62728", lw=2)
plt.axvline(0, color="black", lw=2, label="Hard Wall")
plt.title("Quantum Reflection & Interference at Hard Wall")
plt.xlabel("Position $x$")
plt.ylabel("Probability Density")
plt.legend()
plt.show()
```
</details>

---

### Exercise 2: Quantum Revivals in an Infinite Square Well
In the infinite square well $x \in [0, L]$, the energy eigenvalues are quadratic in $n$:
$$E_n = \frac{n^2 \pi^2 \hbar^2}{2m L^2}$$
Because all energies are integer multiples of the fundamental frequency $\omega_1 = E_1/\hbar$, **any arbitrary initial wave packet will eventually reform its exact original shape** at the revival time:
$$T_{\text{rev}} = \frac{4m L^2}{\pi \hbar}$$
Prove that at half the revival time $t = T_{\text{rev}}/2$, the wave packet splits into two mirror "Schrödinger cat" replicas!


---
## 7. Summary & Completion of Module 0!

### Key Takeaways from Module 0
1. **Wavefunctions to Operators:** Continuous space wave mechanics $\psi(x)$ and discrete algebraic state vectors $|\psi\rangle$ are complementary representations of the same underlying quantum reality.
2. **The Split-Operator Fourier Method:** Provides an exact, unconditionally stable, $\mathcal{O}(N \log N)$ algorithm for solving the time-dependent Schrödinger equation.
3. **Dispersion vs. Confinement:** Free particles disperse inevitably ($\Delta x \propto t$), whereas quadratic harmonic potentials balance dispersion to enable non-spreading **Coherent States**.
4. **Coherent States are Classical-like:** An oscillating coherent state retains its minimum uncertainty $\Delta x \Delta p = \hbar/2$ and follows Newton's equations of motion $\langle x(t) \rangle = x_0 \cos(\omega t)$.
5. **The Bridge to Quantum Optics:** Every mode of the quantized electromagnetic field is a harmonic oscillator. Its ground state is the vacuum, its discrete excitations are photons, its displaced states are coherent laser beams, and its breathing states are squeezed light.

---

### Congratulations on completing Module 0!

You are now fully equipped to begin **Module 1: Single-Mode Quantum States of Light**:
* **Notebook 01:** Fock States, Ladder Operators & Hilbert Space Truncation Convergence
* **Notebook 02:** Coherent States, Displacement Operators & Poissonian Statistics
* **Notebook 03:** Squeezed Light, Quadrature Noise Ellipses & Sub-Poissonian Statistics
* **Notebook 04:** Thermal States, Density Matrices & Bose-Einstein Distributions
* **Notebook 05:** Mandel $Q$-Parameter & Non-Classicality Benchmarking
* **Notebook 06:** Second-Order Coherence $g^{(2)}(0)$: Photon Bunching vs. Anti-Bunching

---
### References & Recommended Reading
* **Erwin Schrödinger**, *Der stetige Übergang von der Mikro- zur Makromechanik*, Naturwissenschaften 14, 664 (1926).
* **Roy J. Glauber**, *Coherent and Incoherent States of the Radiation Field*, Phys. Rev. 131, 2766 (1963) *(Nobel Prize 2005)*.
* **David J. Griffiths**, *Introduction to Quantum Mechanics*, Chapter 2: The Free Particle & The Harmonic Oscillator.
